# Is there enough data to compute the SPB residual? A direct check

This notebook answers one question precisely: **for the materials where DFT and experimental data overlap,
can we actually compute S_SPB and a real residual (S_exp - S_SPB), and is the resulting sample count large
enough to run symbolic regression on -- pooled, per chemical family, and per processing route?**

It reuses the verified data-loading and formula-cleaning code from the first exploration notebook (zip
handling for teMatDb272/ChemDataExtractor, the D/T-isotope + abbreviation filter for composition matching),
then adds the part that notebook didn't do: actually solving the SPB model numerically and joining it to
real measured Seebeck values near the DFT's fixed condition, material by material.

Run top to bottom. Everything downloads live.


## 0. Setup


In [1]:
!pip install -q matminer pymatgen requests pandas scipy

import requests
import pandas as pd
import numpy as np
import io
import re
import json
import zipfile
import ast
from scipy.integrate import quad
from scipy.optimize import brentq
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 797.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.1/829.1 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 47.9 MB/s eta 0:00:00
   ━━━━━

## 1. Load the DFT/BoltzTraP set and confirm its exact fixed condition

The matminer `boltztrap_mp` loader gives a single-point slice of Ricci et al. (2017)'s dataset (confirmed in
the first notebook: no temperature or doping columns). Before computing anything, print matminer's own
dataset description so the fixed (n, T) point this slice corresponds to is confirmed from source, not assumed.


In [2]:
from matminer.datasets import load_dataset, get_all_dataset_info

print(get_all_dataset_info('boltztrap_mp'))


Dataset: boltztrap_mp
Description: Effective mass and thermoelectric properties of 8924 compounds in The  Materials Project database that are calculated by the BoltzTraP software package run on the GGA-PBE or GGA+U density functional theory calculation results. The properties are reported at the temperature of 300 Kelvin and the carrier concentration of 1e18 1/cm3.
Columns:
	formula: Chemical formula of the entry
	m_n: n-type/conduction band effective mass. Units: m_e where m_e is the mass of an electron; i.e. m_n is a unitless ratio
	m_p: p-type/valence band effective mass.
	mpid: Materials Project identifier
	pf_n: n-type thermoelectric power factor in uW/cm2.K where uW is microwatts and a constant relaxation time of 1e-14 assumed.
	pf_p: p-type power factor in uW/cm2.K
	s_n: n-type Seebeck coefficient in micro Volts per Kelvin
	s_p: p-type Seebeck coefficient in micro Volts per Kelvin
	structure: pymatgen Structure object describing the crystal structure of the material
Num Entries:

In [3]:
df_dft = load_dataset('boltztrap_mp')
print('Shape:', df_dft.shape)
print(df_dft.columns.tolist())
df_dft.head()


Fetching boltztrap_mp.json.gz from https://ndownloader.figshare.com/files/13297511 to /usr/local/lib/python3.13/dist-packages/matminer/datasets/boltztrap_mp.json.gz


Fetching https://ndownloader.figshare.com/files/13297511 in MB: 14.020608MB [00:00, 218.74MB/s]                         


Shape: (8924, 9)
['mpid', 'pf_n', 'pf_p', 's_n', 's_p', 'formula', 'm_n', 'm_p', 'structure']


,mpid,pf_n,pf_p,s_n,s_p,formula,m_n,m_p,structure
0,mp-10070,0.865,0.0125,-421.0,760.0,BaAg(PO3)3,0.578,72.800,"[[ 0.08245398 10.58009491 11.61923254] O, [3.1..."
1,mp-10086,1.050,0.6440,-393.0,567.0,YSF,0.541,4.020,"[[2.84699546 0.94899849 0. ] F, [0.9489..."
2,mp-10096,0.977,0.4200,-305.0,471.0,Na3Sr3GaP4,0.270,1.020,"[[-2.8085287 7.06608376 1.25800196] Na, [1...."
3,mp-10103,0.889,0.0257,-416.0,871.0,YbLiAlF6,0.548,20.300,"[[2.52287112 1.45658029 7.18290524] Li, [-2.85..."
4,mp-10155,0.719,1.5700,-496.0,510.0,P2Ir,1.160,0.667,"[[0.86371064 5.14343422 4.80450637] P, [4.5512..."


**Read the printed description above and set these two values by hand before continuing** -- they are the
fixed doping level and temperature this slice was computed at (per Ricci et al. 2017 and the matminer
catalog entry, this is expected to be n = 1e18 cm^-3, T = 300 K, but confirm from the text actually printed
above rather than trusting this comment).


In [4]:
FIXED_N_CM3 = 1e18   # <-- confirm/adjust from the dataset description printed above
FIXED_T_K = 300.0    # <-- confirm/adjust from the dataset description printed above
print(f'Using fixed condition: n = {FIXED_N_CM3:.0e} cm^-3, T = {FIXED_T_K} K')


Using fixed condition: n = 1e+18 cm^-3, T = 300.0 K


## 2. Load teMatDb272 (zip-based, as verified in the first notebook)


In [5]:
repo_api = 'https://api.github.com/repos/byungkiryu/teMatDb/contents/teMatDb_publication'
resp = requests.get(repo_api, timeout=30)
listing = resp.json()

zip_entry = next((f for f in listing if isinstance(listing, list) and f['name'].lower().endswith('.zip')), None)
tematdb_tables = {}
if zip_entry:
    r = requests.get(zip_entry['download_url'], timeout=120)
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    for n in zf.namelist():
        if n.lower().endswith('.csv'):
            with zf.open(n) as fh:
                try:
                    tematdb_tables[n] = pd.read_csv(fh)
                except Exception as e:
                    print(n, 'FAILED to parse:', e)

samples_key = next((k for k in tematdb_tables if 'sample' in k.lower()), None)
collocated_key = next((k for k in tematdb_tables if 'collocated' in k.lower()), None)
df_samples = tematdb_tables.get(samples_key)
df_collocated = tematdb_tables.get(collocated_key)
print('samples:', df_samples.shape if df_samples is not None else None)
print('collocated TEPs:', df_collocated.shape if df_collocated is not None else None)
display(df_collocated.head())


samples: (272, 12)
collocated TEPs: (56641, 6)


,sample_id,Temperature,alpha,rho,kappa,ZT_author_declared
0,1,300,0.000063,0.000004,3.057198,0.099456
1,1,302,0.000064,0.000004,3.036914,0.103578
2,1,304,0.000065,0.000004,3.016630,0.107700
3,1,306,0.000066,0.000004,2.996346,0.111821
4,1,308,0.000067,0.000004,2.976063,0.115943


## 3. Load the ChemDataExtractor thermoelectric DB (zip-based, as verified in the first notebook)


In [6]:
fig_api = 'https://api.figshare.com/v2/articles/19658787'
resp = requests.get(fig_api, timeout=30)
article = resp.json()
files = article.get('files', [])
zip_candidates = [f for f in files if f['name'].lower().endswith('.zip')]

def load_csvs_from_zip_url(url, timeout=120):
    r = requests.get(url, timeout=timeout)
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    tables = {}
    for n in zf.namelist():
        if n.lower().endswith('.csv'):
            with zf.open(n) as fh:
                try:
                    tables[n] = pd.read_csv(fh)
                except Exception as e:
                    print(n, 'FAILED to parse:', e)
    return tables

cde_tables = {}
te_db_zip = next((f for f in zip_candidates if 'TE_databases' in f['name']), None)
if te_db_zip:
    cde_tables = load_csvs_from_zip_url(te_db_zip['download_url'])
if not cde_tables:
    other_zip = next((f for f in zip_candidates if f is not te_db_zip), None)
    if other_zip:
        cde_tables = load_csvs_from_zip_url(other_zip['download_url'])

df_cde = None
if cde_tables:
    main_key = max(cde_tables, key=lambda k: cde_tables[k].shape[0])
    df_cde = cde_tables[main_key]
    print('Using', main_key, ':', df_cde.shape)

resolution_col = next((c for c in df_cde.columns if 'resolution' in c.lower()), None) if df_cde is not None else None
name_col = next((c for c in df_cde.columns if c.lower() == 'name'), None) if df_cde is not None else None
specifier_col = next((c for c in df_cde.columns if c.lower() == 'specifier'), None) if df_cde is not None else None
value_col = next((c for c in df_cde.columns if c.lower() == 'value'), None) if df_cde is not None else None
units_col = next((c for c in df_cde.columns if c.lower() == 'units'), None) if df_cde is not None else None
temp_col = next((c for c in df_cde.columns if c.lower() == 'temperature_average'), None) if df_cde is not None else None
print('Key columns:', resolution_col, name_col, specifier_col, value_col, units_col, temp_col)


Using TE_databases/CSV/main_tedb.csv : (22805, 24)
Key columns: Resolution Name Specifier Value Units Temperature_Average


## 4. Cleaned composition overlap (same filter as the first notebook: D/T-isotope + abbreviation blocklist)


In [7]:
from pymatgen.core.composition import Composition

NON_FORMULA_LABELS = {'HH', 'SKD', 'M2Q', 'ABQ2', 'Cu2Q', 'clathrate', 'etc', 'CNT'}
ISOTOPE_SYMBOLS = {'D', 'T'}

def to_reduced_formula(s, exclude_labels=NON_FORMULA_LABELS):
    if not isinstance(s, str) or not s.strip():
        return None
    if s.strip() in exclude_labels:
        return None
    try:
        comp = Composition(s)
        if any(el.symbol in ISOTOPE_SYMBOLS for el in comp.elements):
            return None
        return comp.reduced_formula
    except Exception:
        return None

dft_formulas = set(df_dft['formula'].map(to_reduced_formula).dropna())

tematdb_formulas = set()
if df_samples is not None and 'BASEMAT' in df_samples.columns:
    tematdb_formulas = set(df_samples['BASEMAT'].map(to_reduced_formula).dropna())
overlap_tematdb = dft_formulas & tematdb_formulas
print('Cleaned overlap DFT <-> teMatDb272:', len(overlap_tematdb), sorted(overlap_tematdb))

def extract_host(s):
    if not isinstance(s, str):
        return None
    m = re.search(r"host['\"]?\s*[:=]\s*['\"]?([A-Za-z0-9().]+)", s)
    return m.group(1) if m else None

cde_host_series = None
if df_cde is not None:
    if resolution_col and df_cde[resolution_col].notna().any():
        cde_host_series = df_cde[resolution_col].map(extract_host)
    if cde_host_series is None or cde_host_series.notna().sum() == 0:
        cde_host_series = df_cde[name_col]

cde_formulas = set(cde_host_series.map(to_reduced_formula).dropna()) if cde_host_series is not None else set()
overlap_cde = dft_formulas & cde_formulas
print()
print('Cleaned overlap DFT <-> ChemDataExtractor:', len(overlap_cde), sorted(overlap_cde)[:40])


/usr/lib/python3.13/functools.py:1025: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


Cleaned overlap DFT <-> teMatDb272: 9 ['CoSb3', 'CuBiSeO', 'GeTe', 'In2O3', 'PbS', 'PbSe', 'SnS', 'SnSe', 'TePb']

Cleaned overlap DFT <-> ChemDataExtractor: 41 ['AgBi3S5', 'AlCuO2', 'AlN', 'BaCeO3', 'BaTiO3', 'Bi2O3', 'Bi2S3', 'CdSe', 'CoSb3', 'CrCuO2', 'Cu2S', 'CuBiSeO', 'CuBr', 'GaN', 'Ge', 'GeSe', 'GeTe', 'H2O2', 'In2O3', 'KGaSb4', 'MoS2', 'NaHO', 'PbS', 'PbSe', 'Sb2Se3', 'Si', 'SiC', 'SnO2', 'SnS', 'SnSe', 'Sr3Ti2O7', 'SrMnO3', 'SrTe', 'TePb', 'TiO2', 'TlInSe2', 'WO3', 'YPO4', 'Zn3In2O6', 'ZnO']


/usr/lib/python3.13/functools.py:1025: UserWarning: No Pauling electronegativity for Ts. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  val = self.func(instance)


## 5. The SPB model itself

Standard acoustic-phonon-limited (r = -1/2) single parabolic band Pisarenko relation, in the same form used
in Naithani & Dasgupta (2020, DOI 10.1021/acsaem.9b02015, Table 1) and de Boor (2021, DOI 10.1016/j.jmat.2020.10.013):

```latex
n = 4\pi \left(\frac{2 m^* k_B T}{h^2}\right)^{3/2} F_{1/2}(\eta) \qquad\qquad
S_{SPB} = \pm\frac{k_B}{e}\left(\frac{2F_1(\eta)}{F_0(\eta)} - \eta\right)
```

where `F_j(eta) = integral_0^inf [x^j / (1 + exp(x - eta))] dx` is the Fermi-Dirac integral of order j, eta is
the reduced Fermi level (solved numerically from the carrier concentration n and the DFT-derived
density-of-states effective mass m*), and the sign of S is negative for n-type (electron) conduction,
positive for p-type (hole) conduction, fixed by convention.

This cell implements it directly and sanity-checks it against a textbook non-degenerate limit
(eta -> -infinity should give the Mott/Boltzmann limit).


In [8]:
M_E = 9.10938e-31      # electron rest mass, kg
K_B = 1.380649e-23     # Boltzmann constant, J/K
H_PLANCK = 6.62607015e-34  # Planck constant, J*s
K_B_OVER_E_UVK = 86.17333262  # k_B/e in microvolt/K

def fermi_dirac_integral(eta, j):
    integrand = lambda x: x**j / (1.0 + np.exp(np.clip(x - eta, -700, 700)))
    val, _ = quad(integrand, 0, 200, limit=200)
    return val

def solve_eta_from_n(n_cm3, m_star_ratio, T_K):
    """Solve for the reduced Fermi level eta given carrier concentration (cm^-3), DOS effective mass
    ratio (m*/m_e, dimensionless, as reported by matminer's m_n/m_p), and temperature (K)."""
    m_star = m_star_ratio * M_E
    n_si = n_cm3 * 1e6  # cm^-3 -> m^-3
    prefactor = 4 * np.pi * (2 * m_star * K_B * T_K / H_PLANCK**2)**1.5
    def f(eta):
        return prefactor * fermi_dirac_integral(eta, 0.5) - n_si
    # bracket a wide range; SPB systems of interest sit roughly in eta in [-30, 30]
    return brentq(f, -60, 60, xtol=1e-6)

def spb_seebeck_uvk(eta, carrier_sign):
    """carrier_sign: -1 for n-type (electrons), +1 for p-type (holes)."""
    F0 = fermi_dirac_integral(eta, 0)
    F1 = fermi_dirac_integral(eta, 1)
    magnitude = K_B_OVER_E_UVK * (2 * F1 / F0 - eta)
    return carrier_sign * magnitude

# Sanity check: strongly non-degenerate limit (eta very negative) should approach the classical/Mott limit,
# where S ~ (k_B/e)*(2 - eta) grows large and 2F1/F0 -> 2 - eta since F_j(eta) ~ exp(eta)*Gamma(j+1) there.
test_eta = -10.0
print('Sanity check at eta = -10 (should be close to classical/non-degenerate limit):')
print('  2F1/F0 - eta =', 2*fermi_dirac_integral(test_eta,1)/fermi_dirac_integral(test_eta,0) - test_eta,
      ' (expect close to 2.0, the classical value, independent of eta in this limit)')


Sanity check at eta = -10 (should be close to classical/non-degenerate limit):
  2F1/F0 - eta = 12.000022699564127  (expect close to 2.0, the classical value, independent of eta in this limit)


## 6. Compute S_SPB and the residual for every teMatDb272 sample in the cleaned overlap

Carrier type isn't given as an explicit column in teMatDb272 -- infer it from the *sign* of the measured
Seebeck coefficient itself (negative = n-type/electrons, positive = p-type/holes, by the standard
convention both datasets follow), then pick m_n or m_p from the DFT row accordingly. S_exp is taken from
`teMatDb_collocatedTEPs.csv` at the temperature closest to the DFT's fixed T (printed per-row so you can see
how far off 300 K the nearest real measurement actually is -- a large gap there would make the comparison
questionable even if a "match" exists).


In [9]:
results_tematdb = []

if df_samples is not None and df_collocated is not None and overlap_tematdb:
    id_col = 'sample_id'
    df_samples = df_samples.copy()
    df_samples['reduced_formula'] = df_samples['BASEMAT'].map(to_reduced_formula)
    dft_by_formula = df_dft.set_index('formula') if df_dft['formula'].is_unique else None

    for _, srow in df_samples[df_samples['reduced_formula'].isin(overlap_tematdb)].iterrows():
        sid = srow[id_col]
        f = srow['reduced_formula']
        curve = df_collocated[df_collocated['sample_id'] == sid]
        if curve.empty or 'alpha' not in curve.columns:
            continue
        curve = curve.copy()
        curve['T_dist'] = (curve['Temperature'] - FIXED_T_K).abs()
        nearest = curve.sort_values('T_dist').iloc[0]
        s_exp_uvk = nearest['alpha'] * 1e6  # teMatDb272 stores alpha in V/K based on the ~1e-4 scale seen earlier; verify below
        T_nearest = nearest['Temperature']

        dft_rows = df_dft[df_dft['formula'].map(to_reduced_formula) == f]
        if dft_rows.empty:
            continue
        dft_row = dft_rows.iloc[0]

        carrier_sign = -1 if s_exp_uvk < 0 else 1
        m_star = dft_row['m_n'] if carrier_sign == -1 else dft_row['m_p']
        if not np.isfinite(m_star) or m_star <= 0:
            continue
        try:
            eta = solve_eta_from_n(FIXED_N_CM3, m_star, FIXED_T_K)
            s_spb_uvk = spb_seebeck_uvk(eta, carrier_sign)
        except Exception as e:
            continue

        results_tematdb.append({
            'sample_id': sid, 'formula': f, 'GROUP': srow.get('GROUP'), 'SINTERING': srow.get('SINTERING'),
            'T_nearest_K': T_nearest, 'T_offset_from_fixed': abs(T_nearest - FIXED_T_K),
            'S_exp_uVK': s_exp_uvk, 'S_SPB_uVK': s_spb_uvk, 'residual_uVK': s_exp_uvk - s_spb_uvk,
            'carrier_type': 'n' if carrier_sign == -1 else 'p', 'm_star_used': m_star,
        })

df_residual_tematdb = pd.DataFrame(results_tematdb)
print('teMatDb272: residual computed for', len(df_residual_tematdb), 'samples out of',
      len(df_samples[df_samples['reduced_formula'].isin(overlap_tematdb)]) if df_samples is not None else 0,
      'in the cleaned overlap')
display(df_residual_tematdb.head(20))


teMatDb272: residual computed for 91 samples out of 91 in the cleaned overlap


,sample_id,formula,GROUP,SINTERING,T_nearest_K,T_offset_from_fixed,S_exp_uVK,S_SPB_uVK,residual_uVK,carrier_type,m_star_used
0,1,TePb,PbTe,na,300.0,0.0,62.787234,220.645479,-157.858245,p,0.1600
1,7,TePb,PbTe,HP,302.0,2.0,139.722365,220.645479,-80.923114,p,0.1600
2,10,TePb,PbTe,SPS,308.0,8.0,69.785567,220.645479,-150.859912,p,0.1600
3,11,TePb,PbTe,SPS,308.0,8.0,-63.816990,-218.403727,154.586737,n,0.1570
4,17,TePb,PbTe,HP,302.0,2.0,51.465357,220.645479,-169.180122,p,0.1600
5,18,TePb,PbTe,HP,302.0,2.0,65.942117,220.645479,-154.703362,p,0.1600
6,23,TePb,PbTe,HP,324.0,24.0,74.769000,220.645479,-145.876479,p,0.1600
7,25,PbS,Sulfide,SPS,304.0,4.0,78.588029,1071.019878,-992.431849,p,122.0000
8,28,SnSe,Selenide,na,306.0,6.0,162.336794,295.327373,-132.990578,p,0.2950
9,73,In2O3,Oxide,SPS,332.0,32.0,-94.900446,-110.811305,15.910859,n,0.0574


**Check the `S_exp_uVK` column above by eye before trusting anything downstream.** teMatDb272's raw `alpha`
values looked like ~1e-4 scale in the first notebook's preview (consistent with V/K, hence the `*1e6` above
to get microvolt/K) -- but confirm the printed magnitudes above land in a sane thermoelectric range
(roughly -1000 to +1000 uV/K) rather than being off by a further factor of 1000, and fix the conversion
factor in the cell above if not.


## 7. Compute S_SPB and the residual for ChemDataExtractor rows in the cleaned overlap

Same logic, but the source table is messier: `Value` is stored as a string (often a bracketed list like
`"[1.47]"`), units vary by property, and only rows where `Specifier` clearly means "Seebeck coefficient"
should be used. Everything is printed for a visual sanity check rather than assumed correct.


In [10]:
def parse_value_field(v):
    """ChemDataExtractor's Value column is usually a stringified list, e.g. '[1.47]' or '[1.2, 1.3]'.
    Return the mean of whatever numbers are found, or None."""
    if not isinstance(v, str):
        return None
    try:
        parsed = ast.literal_eval(v)
        if isinstance(parsed, (list, tuple)) and len(parsed) > 0:
            nums = [x for x in parsed if isinstance(x, (int, float))]
            return float(np.mean(nums)) if nums else None
        if isinstance(parsed, (int, float)):
            return float(parsed)
    except Exception:
        pass
    m = re.findall(r'-?\d+\.?\d*', v)
    return float(m[0]) if m else None

results_cde = []

if df_cde is not None and specifier_col and overlap_cde:
    seebeck_mask = df_cde[specifier_col].astype(str).str.contains('seebeck', case=False, na=False)
    df_seebeck = df_cde[seebeck_mask].copy()
    print('Rows with Specifier containing "seebeck":', len(df_seebeck))
    print('Distinct Units strings seen for these rows (check the conversion factor below against this):')
    print(df_seebeck[units_col].value_counts().head(10) if units_col else 'no Units column found')

    df_seebeck['host_formula'] = cde_host_series.loc[df_seebeck.index].map(to_reduced_formula)
    df_seebeck = df_seebeck[df_seebeck['host_formula'].isin(overlap_cde)]
    df_seebeck['S_exp_parsed'] = df_seebeck[value_col].map(parse_value_field)
    df_seebeck['T_offset_from_fixed'] = (df_seebeck[temp_col] - FIXED_T_K).abs() if temp_col else np.nan

    for _, row in df_seebeck.dropna(subset=['S_exp_parsed']).iterrows():
        f = row['host_formula']
        dft_rows = df_dft[df_dft['formula'].map(to_reduced_formula) == f]
        if dft_rows.empty:
            continue
        dft_row = dft_rows.iloc[0]

        # NOTE: verify against the printed Units above -- assuming values are already in microvolt/K here;
        # adjust the *1 factor below (e.g. to *1e6 or /1e6) once you see the real Units strings.
        s_exp_uvk = row['S_exp_parsed'] * 1

        carrier_sign = -1 if s_exp_uvk < 0 else 1
        m_star = dft_row['m_n'] if carrier_sign == -1 else dft_row['m_p']
        if not np.isfinite(m_star) or m_star <= 0:
            continue
        try:
            eta = solve_eta_from_n(FIXED_N_CM3, m_star, FIXED_T_K)
            s_spb_uvk = spb_seebeck_uvk(eta, carrier_sign)
        except Exception:
            continue

        results_cde.append({
            'formula': f, 'S_exp_uVK': s_exp_uvk, 'S_SPB_uVK': s_spb_uvk,
            'residual_uVK': s_exp_uvk - s_spb_uvk, 'carrier_type': 'n' if carrier_sign == -1 else 'p',
            'T_offset_from_fixed': row['T_offset_from_fixed'], 'DOI': row.get('DOI'),
        })

df_residual_cde = pd.DataFrame(results_cde)
print()
print('ChemDataExtractor: residual computed for', len(df_residual_cde), 'rows')
display(df_residual_cde.head(20))


Rows with Specifier containing "seebeck": 1409
Distinct Units strings seen for these rows (check the conversion factor below against this):
Units
Kelvin^(-1.0)  Volt^(1.0)    1409
Name: count, dtype: int64

ChemDataExtractor: residual computed for 29 rows


,formula,S_exp_uVK,S_SPB_uVK,residual_uVK,carrier_type,T_offset_from_fixed,DOI
0,TePb,-0.000041,-218.403727,218.403686,n,153.0,10.1039/c3ta01594g
1,TePb,0.000094,220.645479,-220.645385,p,5.0,10.1016/j.jallcom.2019.152463
2,ZnO,0.000448,553.289342,-553.288894,p,15.0,10.1016/j.mssp.2016.01.013
3,PbSe,0.000248,219.902347,-219.902099,p,423.0,10.1016/j.mtener.2018.06.010
4,PbSe,0.000083,219.902347,-219.902264,p,5.0,10.1016/j.mtener.2018.06.010
5,PbSe,0.000213,219.902347,-219.902134,p,450.0,10.1016/j.mtener.2018.06.010
6,PbSe,0.000280,219.902347,-219.902067,p,600.0,10.1016/j.mtener.2018.06.010
7,PbSe,-0.000270,-216.124238,216.123968,n,573.0,10.1016/j.mtener.2018.06.010
8,PbSe,-0.000186,-216.124238,216.124052,n,573.0,10.1016/j.mtener.2018.06.010
9,Bi2S3,-0.000244,-346.496111,346.495867,n,373.0,10.1007/s10853-019-04008-3


**Same warning as Section 6: check `S_exp_uVK` above lands in a sane range (~-1000 to +1000 uV/K) given the
real Units strings printed above, and fix the conversion factor in the cell if not** -- this is the single
most likely place for a silent unit-scale bug, since ChemDataExtractor's Units field is a free-text string
per row rather than a fixed schema.


## 8. Is there enough data? Sample-size breakdown by pool, family, and process


In [11]:
df_all_residual = pd.concat([
    df_residual_tematdb.assign(source='teMatDb272') if len(df_residual_tematdb) else pd.DataFrame(),
    df_residual_cde.assign(source='ChemDataExtractor') if len(df_residual_cde) else pd.DataFrame(),
], ignore_index=True, sort=False)

print('=== Overall ===')
print('Total usable (material, near-300K) residual points:', len(df_all_residual))
print(df_all_residual['source'].value_counts() if len(df_all_residual) else 'none')
print()

MIN_FOR_SR = 15  # rough rule-of-thumb floor for a free-form GP-SR fit to be numerically meaningful; not a hard law

print('=== By chemical family (teMatDb272 GROUP) ===')
if len(df_residual_tematdb):
    fam_counts = df_residual_tematdb['GROUP'].value_counts()
    print(fam_counts)
    print()
    print('Families with >=', MIN_FOR_SR, 'points (workable for a per-family SR fit):')
    print(fam_counts[fam_counts >= MIN_FOR_SR])
else:
    print('no teMatDb272 residual rows to group')
print()

print('=== By processing route (teMatDb272 SINTERING) ===')
if len(df_residual_tematdb):
    proc_counts = df_residual_tematdb['SINTERING'].value_counts()
    print(proc_counts)
    print()
    print('Processing routes with >=', MIN_FOR_SR, 'points:')
    print(proc_counts[proc_counts >= MIN_FOR_SR])
else:
    print('no teMatDb272 residual rows to group')


=== Overall ===
Total usable (material, near-300K) residual points: 120
source
teMatDb272           91
ChemDataExtractor    29
Name: count, dtype: int64

=== By chemical family (teMatDb272 GROUP) ===
GROUP
PbTe        46
GeTe        22
Selenide    16
SKD          3
Sulfide      2
Oxide        2
Name: count, dtype: int64

Families with >= 15 points (workable for a per-family SR fit):
GROUP
PbTe        46
GeTe        22
Selenide    16
Name: count, dtype: int64

=== By processing route (teMatDb272 SINTERING) ===
SINTERING
HP                  28
SPS                 12
na                   2
SPS 5min, 60 Mpa     1
Name: count, dtype: int64

Processing routes with >= 15 points:
SINTERING
HP    28
Name: count, dtype: int64


In [12]:
print('=== Residual sanity check: distribution ===')
if len(df_all_residual):
    print(df_all_residual['residual_uVK'].describe())
    print()
    print('Rows where the nearest real measurement is >50K away from the fixed DFT condition',
          '(these are the shakiest matches -- consider excluding them):')
    if 'T_offset_from_fixed' in df_all_residual.columns:
        far = df_all_residual[df_all_residual['T_offset_from_fixed'] > 50]
        print(len(far), '/', len(df_all_residual))
else:
    print('no residual rows computed at all -- see Sections 6-7 for where the pipeline stopped producing rows')


=== Residual sanity check: distribution ===
count    120.000000
mean     -37.603935
std      229.289619
min     -992.431849
25%     -151.820774
50%      -67.374166
75%      140.856216
max      450.552804
Name: residual_uVK, dtype: float64

Rows where the nearest real measurement is >50K away from the fixed DFT condition (these are the shakiest matches -- consider excluding them):
14 / 120


## 9. Verdict

Fill this in from the real numbers printed above once you've run it and fixed any unit-conversion issues
flagged in Sections 6-7:

- **Total usable residual points (pooled)**: is it comfortably more than a few dozen? SR needs enough points
  to distinguish a real pattern from overfitting noise, and this number is the honest ceiling on how good
  any global correction can be trusted to be.
- **Per-family counts (Section 8)**: which chemical families clear the `MIN_FOR_SR` floor and which don't?
  Families below it should be reported as "insufficient for a dedicated per-family fit" rather than forced
  into one -- pool them into a smaller number of family groups, or fold them into the general model only.
- **Per-process counts**: same question for SINTERING (SPS/HP/PAS) -- is there enough of each route
  represented within any one family to actually test a process-wise equation, or only enough pooled across
  families?
- **Temperature-offset check**: how many residual points are built from a real measurement far from the
  DFT's fixed 300 K condition? Those are weaker matches and should probably be flagged or dropped rather
  than silently averaged in.

If the honest numbers here turn out too thin for a per-family fit on the matminer single-point DFT slice,
that is itself a real, useful finding: it would mean the fuller multi-point Ricci/BoltzTraP deposit (Dryad
DOI 10.5061/dryad.gn001) is needed before the family-wise part of the plan is executable, not just the
pooled part.
